In [1]:
# Matplotlib 폰트 설정
import matplotlib.pyplot as plt
from matplotlib import font_manager, rc
import matplotlib


# 나눔고딕 폰트 경로 설정
font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
font_manager.fontManager.addfont(font_path)
rc('font', family='NanumGothic', size=12)


# 마이너스 폰트 깨짐 방지
matplotlib.rcParams['axes.unicode_minus'] = False


print("폰트 설정 완료 - NanumGothic 적용됨")


폰트 설정 완료 - NanumGothic 적용됨


In [2]:
import torch
import torch.nn as nn

In [ ]:
# U-Net 생성자 (Generator) 클래스 정의
# 결론: skip connection 모든 up_conv_layer_X에 적용됨
# 디코더에서 전체적인 맥락 유지, 인코더에서 (색상, 질감) 보존해서 복원(skip-connection)

class UNetGenerator(nn.Module):
    def __init__(self, chnls_in=3, chnls_op=3):
        super(UNetGenerator, self).__init__()


    # 인코더(다운샘플링) 블록  >> 특징 추출
    # DownConvBlock 은 해상도를 줄이고 채널을 늘림
        self.down_conv_layer1 = nn.DownConvBlock(chnls_in, 64, norm=False)
        # 초기 블럭은 정규화 제외 (원본에 가깝게)
        self.down_conv_layer2 = nn.DownConvBlock(64, 128)
        self.down_conv_layer3 = nn.DownConvBlock(128, 256)
        # 깊은 레이어에는 채널은 유지(512) 드롭아웃 적용 하여 과적합 방지
        self.down_conv_layer4 = nn.DownConvBlock(256, 512, dropout=0.5)
        self.down_conv_layer5 = nn.DownConvBlock(512, 512, dropout=0.5)
        self.down_conv_layer6 = nn.DownConvBlock(512, 512, dropout=0.5)
        self.down_conv_layer7 = nn.DownConvBlock(512, 512, dropout=0.5)
        # 병목(bottle-neck) 부분 : 해상도 변경 없음. 정규화, 드랍아웃 비활성화/활성화 옵션 적용
        self.down_conv_layer8 = nn.DownConvBlock(512, 512, norm=False, dropout=0.5)


        # 디코더 블록
        # upconvBlock 해상도 늘리고(2배씩) concat 채널 합치고 >> 채널 수 줄이기
        # skip-connection 통해서 encoder 출력 받음
        # upconvblock의 입력 채널 (이전 디코더의 출력채널 + 해당 인코더 출력 채널)
        # 병목 특징 (enc8) 시작 (출력 512)
        self.up_conv_layer1 = nn.UpConvBlock(512,512, dropout=0.5)

        self.up_conv_layer2 = nn.UpConvBlock(1024,512, dropout=0.5)
        # 512(dec1) + 512(enc6) >> 1024 >> 1024를 받아서 512 출력

        self.up_conv_layer3 = nn.UpConvBlock(1024,512, dropout=0.5)
        # 512(dec2) + 512(enc5) >> 1024 >> 1024를 받아서 512 출력

        self.up_conv_layer4 = nn.UpConvBlock(1024,256, dropout=0.5)
        # 512(dec3) + 512(enc4) >> 1024 >> 1024를 받아서 256 출력
        
        self.up_conv_layer5 = nn.UpConvBlock(512,128)
        # 256(dec4) + 256(enc3) >> 512 >> 512를 받아서 128 출력

        self.up_conv_layer6 = nn.UpConvBlock(256,64)
        # 128(dec5) + 128(enc2) >> 256 >> 256를 받아서 64 출력

        self.up_conv_layer7 = nn.UpConvBlock(128,64)
        # 64(dec6) + 64(enc1) >> 128 >> 128를 받아서 64 출력
        
		# 최종 출력 레이어 정의
        # 최종 해상도 복원하기 위한 업샘플링 레이어 정의
        self.upsample_layer = nn.Upsample(scale_factor=2)
        # 최종 합성곱 전에 패딩을 위한 레이어 정의
        self.zero_pad = nn.ZeroPad2d((1,0,1,0))
        # nn.ZeroPad2d((left, right, top, down)) px에 0으로 채워라


        # 최종 출력 채널(chnls_op)을 맞추기 위한 합성곱 레이어 정의
        self.conv_layer1 = nn.Conv2d(64, chnls_op, 4, padding=1)
        # 입력 채널 64, chnls_op 3(RGB), 4: kernel_size


        # 최종 출력 픽셀 값 [-1,1] 범위 제한 >> Tanh 활성화함수
        self.activation = nn.Tanh()


In [ ]:
# U-NET 생성자 (Generator) 클래스 정의
# 결론: skip connection 모든 up_conv_layer_X 에 적용됨
# 디코더에서 전체적인 맥락 유지, 인코더에서 (색상, 질감) 보존해서 복원(skip-connection)
# U Net 핵심: 이전 디코더 출력 + skip connection(대칭되는 인코더 출력)


class UNetGenerator(nn.Module):
    def __init__(self, chnls_in=3, chnls_op=3):
        super(UNetGenerator, self).__init__()


        # 인코더(다운샘플링) 블록  >> 특징 추출
        # DownConvBlock 은 해상도를 줄이고 채널을 늘림
        self.down_conv_layer1 = nn.DownConvBlock(chnls_in, 64, norm=False)
        # 초기 블럭은 정규화 제외 (원본에 가깝게)
        self.down_conv_layer2 = nn.DownConvBlock(64, 128)
        self.down_conv_layer3 = nn.DownConvBlock(128, 256)
        # 깊은 레이어에는 채널은 유지(512) 드롭아웃 적용 하여 과적합 방지
        self.down_conv_layer4 = nn.DownConvBlock(256, 512, dropout=0.5)
        self.down_conv_layer5 = nn.DownConvBlock(512, 512, dropout=0.5)
        self.down_conv_layer6 = nn.DownConvBlock(512, 512, dropout=0.5)
        self.down_conv_layer7 = nn.DownConvBlock(512, 512, dropout=0.5)
        # 병목(bottle-neck) 부분 : 해상도 변경 없음. 정규화, 드랍아웃 비활성화/활성화 옵션 적용
        self.down_conv_layer8 = nn.DownConvBlock(512, 512, norm=False, dropout=0.5)
      
        # 디코더 (업샘플링) 블록
        # UpConvBlock 해상도 늘리고(2배씩) concat 채널 합치고 >> 채널 수 줄이기
        # skip-connection 통해서 encoder 출력 받음
        # UpConvBlock의 입력 채널 (이전 디코더의 출력채널 + 해당 인코더 출력 채널)


        # 병목 특징(enc8) 시작 (출력 512)
        self.up_conv_layer1 = nn.UpConvBlock(512,512, dropout=0.5)


        self.up_conv_layer2 = nn.UpConvBlock(1024,512, dropout=0.5)
        # 512(dec1) + 512(enc7) >> 1024 >> 1024를 입력 받아서 512 출력
        self.up_conv_layer3 = nn.UpConvBlock(1024,512, dropout=0.5)
        # 512(dec2) + 512(enc6) >> 1024를 입력 받아서 512 출력
        self.up_conv_layer4 = nn.UpConvBlock(1024,512, dropout=0.5)
        # 512(dec3) + 512(enc5) >> 1024를 입력 받아서 512 출력
        self.up_conv_layer5 = nn.UpConvBlock(1024,256, dropout=0.5)
        # 512(dec4) + 512(enc4) >> 1024를 입력 받아서 256 출력
        self.up_conv_layer6 = nn.UpConvBlock(512,128)
        # 256(dec5) + 256(enc3) >> 512를 입력 받아서 128 출력
        self.up_conv_layer7 = nn.UpConvBlock(256,64)
        # 128(dec6) + 128(enc2) >> 256를 입력 받아서 64 출력
        self.up_conv_layer8 = nn.UpConvBlock(128,64)
        # 64(dec7) + 64(enc1) >> 128를 입력 받아서 64 출력

        # 최종 출력 레이어 정의
        # 최종 해상도 복원하기 위한 업샘플링 레이어 정의
        # self.upsample_layer = nn.Upsample(scale_factor=2)
        # 최종 합성곱 전에 패딩을 위한 레이어 정의
        self.zero_pad = nn.ZeroPad2d((1,0,1,0))
        # nn.ZeroPad2d((left, right, top, down)) px에 0으로 채워라


        # 최종 출력 채널(chnls_op)을 맞추기 위한 합성곱 레이어 정의
        self.conv_layer1 = nn.Conv2d(64, chnls_op, 4, padding=1)
        # 입력 채널 64, chnls_op 3(RGB), 4: kernel_size


        # 최종 출력 픽셀 값 [-1,1] 범위 제한 >> Tanh 활성화함수
        self.activation = nn.Tanh()


    def forward(self,x):
        
